# GOL-09 — Perplexite structurelle Hashlife : instrument, mesure, et la question Turing-complet

**Navigation** : [<< GOL-08 Translateur Loi II](GOL-08-Loi-II-Translateur-Life.ipynb) · [Index](../README.md) · [>> Lean-17 Conway Bridge to Knots](../Lean-17-Conway-Bridge-to-Knots-Python.ipynb)

Ce carnet consomme l'instrument `scripts/hashlife/k_trajectory.py` et ses resultats `scripts/hashlife/k_trajectory_results.json`, puis examine la these du pivot probabiliste : la dissolution par fragilite (soupe) versus l'auto-entretien (programme recursif) se distinguent-elles dans la mesure `K_trajectory(t, W) = W` ?

**Le verdict tranche 1, important.** L'instrument collapse les programmes Conway simples (glider, blinker, pulsar, OTCA static) au-dela du seuil 0.7 du discriminant, les classant `PROGRAM-PERIODIC-COLLAPSED` -- la periode courte est detectee par LZ, et le quotient `K(W=16)/K(W=1)` plafonne a 0.15. Les trajectoires de soupe ont un quotient entre 0.76 et 0.78 (classe `SOUP-FRAGILE-WEAK`), une signature moins compressible que les periodiques mais loin du seuil `PROGRAM-AUTO-ENTRETIEN`. Aucun temoin du corpus courant ne valide la these d'origine. **Tranche 2 discute pourquoi, et propose un test Turing-complet (OTCA replicator, Spartan universal computer) comme banc de confirmation.**

## 1. Chargement des mesures tranche 1

L'instrument est livre par `scripts/hashlife/k_trajectory.py` (PR #18639). Il definit :

- `K_trajectory(t, W)` : longueur compressee LZ (zlib) du flux d'octets de la trajectoire fenetree a `W` colonnes ; c'est une borne superieure de la complexite de Kolmogorov de la trajectoire.
- `lz_compressed_length(data)` : la primitive de compression utilisee.

**Reproducibilite** : `k_trajectory.py --mode measure` est rejouable localement (Python 3.14). Les resultats chiffres ici proviennent de la sortie JSON livree par la tranche 1 -- 7 trajectoires, 5 fenetres W=1,2,4,8,16.

In [1]:
from pathlib import Path
import json

# Racine du depot : depuis GOL/, remonter jusqu'au dossier qui porte scripts/.
ROOT = Path('.').resolve()
while not (ROOT / 'scripts' / 'hashlife').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RESULTS = ROOT / 'scripts' / 'hashlife' / 'k_trajectory_results.json'
print(f'Chargement {RESULTS.name} (depuis scripts/hashlife/)')
data = json.loads(RESULTS.read_text(encoding='utf-8'))
print(f'Trajectoires mesurees : {len(data["results"])}')
_classes = sorted({v.split('(')[0].strip() for v in data['verdicts'].values()})
print(f'Verdicts tranches : {len(data["verdicts"])} trajectoires, {len(_classes)} classes distinctes')
for name, verdict in data['verdicts'].items():
    print(f'  - {name:30s} -> {verdict.split("(")[0].strip()}')


Chargement k_trajectory_results.json (depuis scripts/hashlife/)
Trajectoires mesurees : 35
Verdicts tranches : 7 trajectoires, 2 classes distinctes
  - glider_T4                      -> PROGRAM-PERIODIC-COLLAPSED
  - blinker_T2                     -> PROGRAM-PERIODIC-COLLAPSED
  - pulsar_T3                      -> PROGRAM-PERIODIC-COLLAPSED
  - otca_static                    -> PROGRAM-PERIODIC-COLLAPSED
  - soup_d03_seed42                -> SOUP-FRAGILE-WEAK
  - soup_d05_seed123               -> SOUP-FRAGILE-WEAK
  - soup_d05_seed777               -> SOUP-FRAGILE-WEAK


## 2. Le tableau des mesures brutes

`K_trajectory(t, W)` pour les 7 trajectoires du corpus, fenetre W = 2^n, n = 0..4 (donc W = 1, 2, 4, 8, 16). La trajectoire est codee en flux d'octets (1 bit par cellule, MSB first dans chaque octet, ligne par ligne). Les lignes du tableau sont les programmes periodiques simples (haut) et les soupes (bas).

In [2]:
import pandas as pd

rows = []
for r in data['results']:
    rows.append({
        'trajectoire': r['trajectory'],
        'W': r['W'],
        'K_trajectory': r['k_trajectory'],
    })
df = pd.DataFrame(rows)
pivot_k = df.pivot(index='trajectoire', columns='W', values='K_trajectory')
pivot_k.columns = [f'W={c}' for c in pivot_k.columns]
pivot_k


,W=1,W=2,W=4,W=8,W=16
trajectoire,,,,,
blinker_T2,240,160,92,46,25
glider_T4,280,200,152,82,43
otca_static,352,200,100,50,27
pulsar_T3,437,319,196,101,54
soup_d03_seed42,625,558,519,500,476
soup_d05_seed123,640,568,534,518,486
soup_d05_seed777,643,583,546,524,504


## 3. Le discriminant en echelle log-log

L'idee d'origine (these user 10:46Z sur #18379) : pour une trajectoire recursive, le quotient `K_trajectory(t, W) / W` reste borne inferieureurement par une constante > 0 quand W croit ; pour une trajectoire de soupe, il tend vers 0. Cela se traduit graphiquement par :

- **Programme recursif** : sur un graphe log(K) vs log(W), la courbe reste au-dessus d'une asymptote de pente 1 (K proportionnel a W).
- **Soupe** : la courbe tend vers une pente < 1 (K sous-lineaire en W).



In [3]:
import math

print('--- pentes log-log (delta log K / delta log W) ---')
ws = sorted(pivot_k.columns, key=lambda c: int(c.replace('W=', '')))  # tri numerique : W=1,2,4,8,16
ws_num = [int(c.replace('W=', '')) for c in ws]
for traj in pivot_k.index:
    ks = pivot_k.loc[traj, ws].values
    slopes = []
    for i in range(1, len(ws)):
        slope = (math.log10(ks[i]) - math.log10(ks[i-1])) / (math.log10(ws_num[i]) - math.log10(ws_num[i-1]))
        slopes.append(slope)
    print(f'{traj:30s}: pentes = {[round(s, 2) for s in slopes]}')


--- pentes log-log (delta log K / delta log W) ---
blinker_T2                    : pentes = [-0.58, -0.8, -1.0, -0.88]
glider_T4                     : pentes = [-0.49, -0.4, -0.89, -0.93]
otca_static                   : pentes = [-0.82, -1.0, -1.0, -0.89]
pulsar_T3                     : pentes = [-0.45, -0.7, -0.96, -0.9]
soup_d03_seed42               : pentes = [-0.16, -0.1, -0.05, -0.07]
soup_d05_seed123              : pentes = [-0.17, -0.09, -0.04, -0.09]
soup_d05_seed777              : pentes = [-0.14, -0.09, -0.06, -0.06]


### Lecture du resultat -- ce que les pentes mesurees disent

Sur le corpus courant, toutes les pentes mesurees sont **negatives ou quasi nulles** : aucune trajectoire ne suit l'asymptote de pente 1 annoncee par la these. Ce qui discrimine est l'**amplitude de la decroissance**, pas le signe : les periodiques courtes (glider T=4, blinker T=2, pulsar T=3, OTCA static) chutent de -0.4 a -1.0 par doublement de W -- otca_static atteint exactement -1.0 sur W=2 -> W=4, soit log2(100/200) = -1, sa complexite mesuree est inversement proportionnelle a W -- tandis que les soupes restent quasi plates, de -0.04 a -0.17 (soup_d05_seed777 W=2 -> W=4 : log2(546/583) = -0.09). La periode courte est compressible des la fenetre W=2 ; les soupes ne se compressent que marginalement quand W croit.

## 4. Pourquoi les periodiques courts sont classes PROGRAM-PERIODIC-COLLAPSED

Le discriminant officiel tranche 1 (`K(W=16) / K(W=1)` < 0.7) marque `PROGRAM-PERIODIC-COLLAPSED` les motifs dont la trajectoire devient tres compressible a grande fenetre. Cela reflete la periode courte (P2 / P3 / P4), pas une absence d'auto-entretien.

**L'instrument mesure l'entropie LZ, pas l'auto-entretien.**

Un motif periodique de periode p produit un flux d'octets ou le bloc de p colonnes se repete indefiniment : `K_trajectory(t, W)` plafonne tres vite au-dela de W=p (la repetition capture toute l'information dans une copie). A l'inverse, un programme non periodique (mais Turing-complet : OTCA replicator a la periode 3530M, Spartan universal computer) maintiendrait sa complexite a toutes les echelles de W.

Les seuils tranches 1 sont donc :

- `< 0.7` : `PROGRAM-PERIODIC-COLLAPSED` -- motif periodique a periode tres inferieure a W=16 ; l'instrument collapse.
- `[0.7, 0.95)` : `SOUP-FRAGILE-WEAK` -- entropie non negligeable qui ne se reduit que partiellement quand W croit.
- `>= 0.95` : `PROGRAM-AUTO-ENTRETIEN` -- aucune mesure sur le corpus courant.

**Le discriminant ne discrimine pas soupe vs programme.** Il discrimine periodique-court vs non-periodique. Les periodiques courts du corpus sont des cas triviaux qui ne mettent pas en valeur la capacite distinctive d'un programme Turing-complet.

In [4]:
# Visualisation du discriminant pour les 7 trajectoires
discriminants = []
for traj in pivot_k.index:
    k1 = pivot_k.loc[traj, 'W=1']
    k16 = pivot_k.loc[traj, 'W=16']
    ratio = k16 / k1
    verdict = data['verdicts'].get(traj, '')
    discriminants.append({
        'trajectoire': traj,
        'K(W=1)': k1,
        'K(W=16)': k16,
        'ratio K(W=16)/K(W=1)': round(ratio, 4),
        'verdict tranche 1': verdict.split('(')[0].strip(),
    })
disc_df = pd.DataFrame(discriminants)
disc_df


,trajectoire,K(W=1),K(W=16),ratio K(W=16)/K(W=1),verdict tranche 1
0,blinker_T2,240,25,0.1042,PROGRAM-PERIODIC-COLLAPSED
1,glider_T4,280,43,0.1536,PROGRAM-PERIODIC-COLLAPSED
2,otca_static,352,27,0.0767,PROGRAM-PERIODIC-COLLAPSED
3,pulsar_T3,437,54,0.1236,PROGRAM-PERIODIC-COLLAPSED
4,soup_d03_seed42,625,476,0.7616,SOUP-FRAGILE-WEAK
5,soup_d05_seed123,640,486,0.7594,SOUP-FRAGILE-WEAK
6,soup_d05_seed777,643,504,0.7838,SOUP-FRAGILE-WEAK


## 5. Le test Turing-complet qui manque : OTCA replicator et Spartan

L'echec de l'instrument tranche 1 est diagnostique : il n'a pas ete applique a un temoin Turing-complet. Les periodiques courtes sont triviales ; les periodiques longues (milliards de generations) ou non-periodiques (universal computers) sont la cible.

**Trois candidats documentes dans la litterature :**

1. **OTCA metapixel** (Brice Due, 2006) : un meta-automate 2048 x 2048 de periode 35328 qui simule n'importe quel automate cellulaire par inclusion. Sa periode, dans la metapixel, equivaut a celle de l'automate simule. Pour simuler Conway sur 100 generations, la periode OTCA est de l'ordre de 3,5 millions (35328 x 100). Un replicator base sur OTCA produit une periode du meme ordre de grandeur.

2. **Spartan universal constructor** (Adam P. Goucher, 2009) : un macrostructure qui peut etre programmee pour executer n'importe quel algorithme, sans periodicite ferme (la periode depend du programme). Sa trajectoire est non-periodique. L'hypothese qu'une telle trajectoire maximise l'entropie LZ reste **non mesuree** : aucun corpus Spartan n'a ete compresse a ce jour.

3. **Hashlife macrocellulaire** (Bill Gosper, 1984, « Exploiting Regularities in Large Cellular Spaces », Physica D 10:75-80) : une construction hierarchique qui evolue par blocs de 2^(2n) cellules avec memoisation. La trajectoire est une megastructure stable.

Hypothese de travail, **non mesuree** : ces cibles exigeront des corpus volumineux (possiblement plusieurs megaoctets) dont le cout de generation n'a pas encore ete confronte a l'instrument LZ Python local -- la tranche 3 tranchera par la mesure. La tranche 3 (futur) pourrait les servir : OTCA etant le plus accessible (format RLE largement diffuse).

**Justification du verdict CONJECTURE-NON-VERIFIABLE.** Les seuils tranches 1 (< 0.7 / 0.7-0.95 / >= 0.95) operent une discrimination periodique vs entropie, pas programme vs soupe. Sans corpus Turing-complet (OTCA ou Spartan), il est impossible de trancher la these d'origine : le discriminant actuel est trop grossier pour separer auto-entretien reel (Turing-complet) de bruit entropique (soupe stationnaire).

## 6. Verdict falsifiable

Le pivot probabiliste du user (these 10:46Z, #18379) postule que `K_trajectory(t, W) / W` tend vers 0 pour la soupe et reste borne inferieureurement pour les programmes recursifs. **Cette these n'est pas verifiee par le corpus tranche 1.** Les periodiques courts (glider, blinker, pulsar, OTCA static) sont classes `PROGRAM-PERIODIC-COLLAPSED`, et les soupes sont `SOUP-FRAGILE-WEAK`. Aucun temoin du corpus courant n'atteint le seuil `PROGRAM-AUTO-ENTRETIEN`.

**Verdict tranche 2** : `CONJECTURE-NON-VERIFIEE-AVEC-CORPUS-COURANT`

L'instrument `K_trajectory(t, W)` mesure la **compressibilite LZ**, qui discrimine periodique-court vs entropie-stationnaire, mais pas auto-entretien vs dissolution-par-fragilite (le discriminant annonce). Pour discriminer ces deux regimes il faudrait :

1. **Un corpus Turing-complet** (OTCA replicator, Spartan universal computer) pour mesurer le quotient sur une trajectoire non-periodique massivement complexe. Le banc n'est pas accessible dans l'instrument tranche 1 (limite memoire).

2. **Un invariant different -- a condition de distinguer trois notions de complexite**. (a) La **longueur zlib** que mesure l'instrument : bornee par le format de description LZ et les ressources du compresseur. (b) La **complexite de Kolmogorov ideale** : pour TOUT programme deterministe fixe P (regle de Life et graines comprises), le prefixe de T generations se decrit par P et T, donc sa complexite est au plus `K(P) + O(log T)` -- la completude de Turing n'etablit PAS une croissance lineaire incompressible au sens ideal, et `K(t)/T -> 0` pour toute trajectoire deterministe. (c) Une **complexite bornee en ressources** (temps ou memoire de description limites) : seul regime ou la distinction programme vs soupe peut survivre, car decrire la soupe exige de rejouer le calcul integralement tandis qu'un replicateur porte sa propre description courte. La these d'origine n'a de contenu operationnel que dans les regimes (a) et (c), pas au sens ideal (b).

3. **Un test dual reformule en complexite bornee en ressources** : au sens ideal, `K(t) / T -> 0` pour toute trajectoire deterministe (borne `K(P) + O(log T)` ci-dessus) ; la reformulation operationnelle doit donc mesurer une complexite **bornee en ressources** -- longueur du plus court programme reproduisant `t` en temps polynomial en T, ou longueur compresse par LZ sur fenetres glissantes. C'est cette version bornee, et elle seule, qui peut separer auto-entretien (taux d'information maintenu sous contrainte de ressources) de dissolution par fragilite.

## 7. Bornes de taille de programme (tranche 3, #19227)

La tranche 2 a mesure un discriminant (`K(W=16) / K(W=1)`) et conclu `CONJECTURE-NON-VERIFIEE-AVEC-CORPUS-COURANT`. La tranche 3 change de question : plutot que de chercher si le quotient **converge**, elle publie la **borne inferieure de Kolmogorov** de chaque temoin, c'est-a-dire la meilleure compression LZ atteignable a la fenetre la plus large du corpus :

- `K_min(t) = K(t, W_last)` : la borne inferieure LZ, mesuree a `W_last = 16` ;
- `K_first(t) = K(t, W=1)` : la borne superieure triviale (flux brut, une colonne a la fois) ;
- `ratio = K_min / K_first` : la fraction de la description brute qui survit a la compression.

L'instrument porte ce calcul en mode dedie (`scripts/hashlife/k_trajectory.py --mode bounds`, tranche 3, PR #19228). La cellule suivante l'appelle sur les resultats tranche 1 charges en section 1 : aucune mesure n'est rejouee, les bornes sont **derivees** des `K_trajectory(t, W)` deja mesures.


In [5]:
# Bornes inferieures de Kolmogorov (LZ) par temoin -- instrument tranche 3 (#19227).
# Le script est importe directement : c'est le meme code que la CLI
# `python scripts/hashlife/k_trajectory.py --mode bounds --json-in ...`.
import importlib.util

_spec = importlib.util.spec_from_file_location(
    'k_trajectory_bounds', ROOT / 'scripts' / 'hashlife' / 'k_trajectory.py'
)
_kt = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_kt)

bounds = _kt.compute_bounds(data['results'])
final = _kt.final_verdict(bounds, data['verdicts'])

bounds_df = pd.DataFrame([
    {
        'temoin': name,
        'W_first': b['window_first'],
        'W_last': b['window_last'],
        'K_first (borne triviale)': b['k_first_lz'],
        'K_min (borne LZ)': b['k_min_lz'],
        'ratio K_min/K_first': round(b['ratio'], 4),
        'verdict tranche 1': data['verdicts'].get(name, '').split('(')[0].strip(),
    }
    for name, b in bounds.items()
]).sort_values('ratio K_min/K_first')

print(f'Verdict final T12 (#18446) : {final}')
bounds_df


Verdict final T12 (#18446) : NON-VERIFIEE (limites de K_trajectory documentees tranche 2)


,temoin,W_first,W_last,K_first (borne triviale),K_min (borne LZ),ratio K_min/K_first,verdict tranche 1
3,otca_static,1,16,352,27,0.0767,PROGRAM-PERIODIC-COLLAPSED
1,blinker_T2,1,16,240,25,0.1042,PROGRAM-PERIODIC-COLLAPSED
2,pulsar_T3,1,16,437,54,0.1236,PROGRAM-PERIODIC-COLLAPSED
0,glider_T4,1,16,280,43,0.1536,PROGRAM-PERIODIC-COLLAPSED
5,soup_d05_seed123,1,16,640,486,0.7594,SOUP-FRAGILE-WEAK
4,soup_d03_seed42,1,16,625,476,0.7616,SOUP-FRAGILE-WEAK
6,soup_d05_seed777,1,16,643,504,0.7838,SOUP-FRAGILE-WEAK


### Lecture du resultat -- ce que les bornes disent

Le tableau se separe en deux blocs nets, et la frontiere tombe entre les deux :

| Bloc | Temoins | `ratio K_min/K_first` | Ce que la borne LZ dit |
|---|---|---|---|
| Programmes periodiques | `glider_T4`, `blinker_T2`, `pulsar_T3`, `otca_static` | 0.077 a 0.154 | la periode courte est entierement capturee : LZ compresse la description d'un facteur 6 a 13 |
| Soupes | `soup_d03_seed42`, `soup_d05_seed123`, `soup_d05_seed777` | 0.759 a 0.784 | la borne reste haute : environ trois quarts de la description brute survivent a la compression |

**Verdict final de la tranche 3** : `NON-VERIFIEE (limites de K_trajectory documentees tranche 2)`.

Ce verdict n'est pas un echec de l'instrument : c'est son resultat falsifiable. Le ratio separe proprement les deux blocs **sur ce corpus** -- le plus grand ratio de programme (0.154, glider) reste tres en dessous du plus petit ratio de soupe (0.759) -- mais aucun temoin ne franchit le seuil `PROGRAM-AUTO-ENTRETIEN`. Distinguer un programme auto-entretenu d'une soupe qui ne se dissout pas encore demande le corpus Turing-complet discute en section 5.

**Limite de lecture, explicite.** `K_min` est la borne de la **fenetre la plus large disponible** (`W_last = 16`), pas la limite asymptotique : sur un corpus a cinq fenetres, `K(t, W=16)` reste une borne basse par defaut, elle ne mesure pas la complexite de Kolmogorov. Les valeurs absolues ne se comparent donc qu'entre temoins du **meme corpus** ; un corpus plus large deplacerait les deux blocs, et l'ordre relatif est le seul resultat transportable.

**References** : tranche 3 issue #19227 ; T12 `Part of #18446` ; EPIC parent #13483 (Lean).


## 8. Le cout en temps : l'exposant α de HashLife contre la simulation naive

Les sections 3 et 4 ont mesure une **compressibilite** : la pente de `K_trajectory(t, W)` en echelle log-log, puis le classement des trajectoires qui en decoule. La section 7 a borne la **taille de programme** qui survit a la compression. Les trois mesures portent sur la **description** d'une trajectoire.

Cette section pose la meme question par l'autre bout : **combien de temps** coute sa production ? HashLife n'est pas une simulation acceleree du meme calcul ; c'est un algorithme qui **reconnait et reutilise** les sous-arbres identiques. La simulation naive paie chaque generation ; HashLife paie la premiere occurrence, puis, sur un motif qui se repete, ne paie plus que la nouveaute. Si la compressibilite mesuree en section 3 est reelle, elle doit se **traduire en temps de calcul** -- et de la meme facon que la section 3 la mesure : par une **pente**, pas par un chronometre isole.

L'instrument est `scripts/hashlife/hashlife_speedup.py` (Pli 3 de l'EPIC Origami #19742, issue #22227). Trois precautions sont ecrites avant la mesure, et verifiables dans le JSON qu'il produit :

1. **Aucune reimplementation.** L'instrument importe l'organe `conway_lean/scripts/hashlife.py` (`construct` / `advance`) et en consigne le `sha256` : ce qui est chronometre est le moteur prouve, pas une copie.
2. **Accord de population exige.** A chaque point, les deux moteurs doivent rendre le **meme effectif** ; un temps mesure sur deux calculs differents ne compare rien.
3. **Memoisation neutralisee.** L'organe memoise globalement (`lru_cache` sur `join` / `life`), donc dans un meme processus un motif rechauffe le cache du suivant -- et un motif *plus long* peut sortir *plus rapide*. Chaque point tourne donc dans un **sous-processus froid**, et le temps retenu est la **mediane de 3 repetitions**.

L'exposant se lit comme une pente, au sens de la section 3 : `alpha = delta log(temps) / delta log(generations)`. Un algorithme qui paie chaque generation a `alpha` proche de 1 ; un algorithme dont le cout ne depend pas de la duree visee a `alpha` proche de 0.

**Seuils declares avant la mesure**, et non retouches apres lecture : `REDUCIBLE` si `alpha < 0.5`, `IRREDUCIBLE` si `alpha >= 0.8`, `AMBIGUOUS` entre les deux.


In [6]:
# Exposant d'echelle du temps HashLife contre la simulation naive -- Pli 3 (#22227).
# Le JSON est celui qu'a ecrit l'instrument du grain A : aucune valeur n'est recopiee a la main.
SPEEDUP = ROOT / 'scripts' / 'hashlife' / 'hashlife_speedup_results.json'
speedup = json.loads(SPEEDUP.read_text(encoding='utf-8'))

seuils = speedup['thresholds']
print(f"Seuils declares avant la mesure : REDUCIBLE si alpha < {seuils['alpha_reducible_max']}, "
      f"IRREDUCIBLE si alpha >= {seuils['alpha_irreducible_min']}, sinon AMBIGUOUS")
print(f"Repetitions par point : {speedup['repeats']} (sous-processus froid, mediane)")
print(f"Organe chronometre : {speedup['organ']['path']}")
print(f"  sha256 = {speedup['organ']['sha256']}")
env = speedup['environment']
print(f"Environnement de mesure : Python {env['python']} / {env['platform']}")
print()

speedup_df = pd.DataFrame([
    {
        'motif': m['motif'],
        'anatomie': m['anatomy'],
        'alpha HashLife': round(m['alpha_hashlife'], 3),
        'alpha naif': round(m['alpha_naive'], 3),
        'gain final (naif / HashLife)': round(m['final_ratio'], 2),
        'points': len(m['points']),
        'population concorde': m['populations_agree'],
        'verdict': m['verdict'],
    }
    for m in speedup['motifs']
]).sort_values('alpha HashLife')

accords = sum(1 for m in speedup['motifs'] if m['populations_agree'])
print(f"Accord de population : {accords}/{len(speedup['motifs'])} motifs")
print()
print('Points bruts des deux extremes (temps en millisecondes) :')
for nom in ('blinker', 'acorn'):
    m = next(x for x in speedup['motifs'] if x['motif'] == nom)
    print(f"  {nom} ({m['anatomy']}), alpha = {m['alpha_hashlife']:.3f} :")
    for p in m['points']:
        print(f"    G={p['generations']:>6d}   naif {p['naive_s'] * 1e3:9.3f} ms"
              f"   hashlife {p['hashlife_s'] * 1e3:9.3f} ms"
              f"   gain {p['ratio']:8.2f}x")

speedup_df


Seuils declares avant la mesure : REDUCIBLE si alpha < 0.5, IRREDUCIBLE si alpha >= 0.8, sinon AMBIGUOUS
Repetitions par point : 3 (sous-processus froid, mediane)
Organe chronometre : MyIA.AI.Notebooks/SymbolicAI/Lean/conway_lean/scripts/hashlife.py
  sha256 = 233303a1354a1d2735b5a8c5371db11a0a765ad65960cccc48dc392f47b74417
Environnement de mesure : Python 3.14.3 / Windows-11-10.0.26300-SP0

Accord de population : 8/8 motifs

Points bruts des deux extremes (temps en millisecondes) :
  blinker (periodique-p2), alpha = 0.106 :
    G=   256   naif     0.710 ms   hashlife     0.743 ms   gain     0.96x
    G=  1024   naif     2.838 ms   hashlife     0.847 ms   gain     3.35x
    G=  4096   naif    11.329 ms   hashlife     0.981 ms   gain    11.55x
    G= 16384   naif    53.133 ms   hashlife     1.167 ms   gain    45.53x
    G= 65536   naif   201.853 ms   hashlife     1.319 ms   gain   153.02x
  acorn (methuselah-chaotique), alpha = 1.352 :
    G=    64   naif     3.763 ms   hashlife    14.1

,motif,anatomie,alpha HashLife,alpha naif,gain final (naif / HashLife),points,population concorde,verdict
1,blinker,periodique-p2,0.106,1.026,153.02,5,True,REDUCIBLE
5,pulsar,periodique-p3,0.117,0.931,15.37,4,True,REDUCIBLE
2,block,immuable,0.158,1.030,106.92,5,True,REDUCIBLE
3,glider,periodique-translatif-p4,0.160,0.994,95.37,5,True,REDUCIBLE
4,gosper-gun,croissance-reguliere,0.289,1.917,1669.28,4,True,REDUCIBLE
7,soup-64-d30,soupe-chaotique,0.571,0.893,2.07,4,True,AMBIGUOUS
6,soup-128-d35,soupe-chaotique,0.675,0.834,1.17,4,True,AMBIGUOUS
0,acorn,methuselah-chaotique,1.352,1.677,0.72,5,True,IRREDUCIBLE


### Lecture du resultat -- le temps suit la compressibilite

Le classement par `alpha` tombe sur les memes temoins que celui de la section 7 :

| Bloc | Motifs | `alpha` HashLife | Gain final (naif / HashLife) |
|---|---|---|---|
| Ce qui se replie | `blinker`, `pulsar`, `block`, `glider` | 0.106 a 0.160 | 15x a 153x |
| Ce qui pousse regulierement | `gosper-gun` | 0.289 | environ 1669x |
| Ce qui ne se replie pas | `acorn` | 1.352 | 0.72x |
| Soupes chaotiques | `soup-64-d30`, `soup-128-d35` | 0.571 et 0.675 | 2.07x et 1.17x |

Trois lectures, dans l'ordre de ce que les chiffres etablissent.

**1. L'exposant discrimine la ou le temps absolu ne dit rien.** Sur les motifs fixes, le naif a `alpha` entre 0.93 et 1.03 : son cout est proportionnel a la duree visee, ce qui valide l'instrument -- il mesure bien une echelle, pas du bruit machine. Sur ces memes motifs, HashLife tombe sous `alpha = 0.16` : son cout depend de la **nouveaute** rencontree, pas de la duree demandee. Le gain final (jusqu'a 153x) n'est que la consequence de cet ecart de pente, pas une propriete separee.

**2. `acorn` est le temoin negatif decisif.** `alpha = 1.352`, au-dessus du seuil `IRREDUCIBLE` et proche du naif (1.677), avec un gain final de 0.72x : sur un methuselah chaotique, HashLife prend **1.38 fois plus de temps** que la simulation directe -- il paie le hachage de sous-arbres qui ne se repetent jamais. Un motif chaotique n'est donc pas un motif periodique ralenti : c'est une autre classe de cout. Sans ce temoin, l'argument « HashLife accelere » se serait lu sur le seul corpus des periodiques, ou il est vrai par construction.

**3. Les soupes restent `AMBIGUOUS`, et le seuil n'a pas bouge apres lecture.** `alpha = 0.571` et `0.675` tombent entre les deux seuils. Le gain final depasse 1 (2.07x et 1.17x) sans que la pente bascule : une soupe finie gagne un peu, parce qu'elle developpe des sous-structures locales et repetees, sans que l'echelle soit dominee par elles. Requalifier ces deux motifs en `REDUCIBLE` au motif que le gain depasse 1 aurait ete un ajustement post-hoc ; le verdict du JSON est repris tel quel.

**Ce que cela ajoute aux sections 3 et 4.** La section 3 mesurait la pente de `K_trajectory(t, W)` -- la compressibilite **statique** de la description produite -- et la section 4 en tirait un classement. Ici la meme question est posee au **cout de production**. Les motifs que la section 4 classait `PROGRAM-PERIODIC-COLLAPSED` sont exactement ceux dont l'exposant de temps s'effondre, et `acorn`, qui ne se replie pas, est le seul dont le cout depasse celui du naif. Deux mesures independantes -- taille de description, temps de calcul -- se rejoignent sur le meme partage, ce qu'aucune des deux ne pourrait fournir seule.

**Limite de lecture, explicite.** Les temps sont mesures sur une machine et dans un environnement consignes dans le JSON, avec l'empreinte `sha256` de l'organe. Les **secondes absolues** ne se comparent donc pas d'une machine a l'autre ; les **exposants**, oui, tant que la comparaison reste motif contre motif dans le meme corpus. Les points bas d'`acorn` (G=64) durent quelques millisecondes, echelle ou le cout fixe de construction de l'arbre domine : c'est le point haut qui porte la pente, et l'instrument ne pondere pas les points par leur duree.

**References** : Pli 3 de l'EPIC Origami #19742, issue #22227, instrument `scripts/hashlife/hashlife_speedup.py` (grain A, PR #22228).


## 9. Conclusion

L'instrument `scripts/hashlife/k_trajectory.py` est correct : il calcule `K_trajectory(t, W)` et distingue les periodiques courts des entropies moyennes. Mais le **pivot** annonce -- `K_trajectory(t, W) / W -> 0 (soupe) vs constante (programme)` -- n'est pas verifie par le corpus tranche 1. L'instrument n'a pas ete confronte a un temoin Turing-complet.

**Cette tranche 2 documente cet ecart** et propose trois pistes pour tranche 3 (corpus OTCA, invariant conditionnel, moyenne glissante). La **tranche 3** (section 7, issue #19227) a depuis publie les **bornes de taille de programme** et le verdict final `NON-VERIFIEE` : le residu n'est plus l'instrument, mais le **corpus Turing-complet** (megaoctets), qui depend d'un arbitrage user sur la these reformulee.

**La section 8** (Pli 3, issue #22227) ferme la boucle par l'autre bout, le **cout de production** : l'exposant `alpha` du temps HashLife contre la simulation naive suit le meme partage que la compressibilite -- de 0.106 a 0.160 sur les periodiques courts, 0.289 sur le canon de Gosper, 1.352 sur `acorn` ou HashLife est plus lent que la simulation directe, et `AMBIGUOUS` sur les deux soupes avec des seuils declares avant la mesure. Ce que le corpus actuel etablit : **la compressibilite se paie en temps, et le partage entre motifs se replie sur deux instruments independants**. Ce qu'il n'etablit toujours pas : la frontiere avec un programme auto-entretenu, qui demande le corpus Turing-complet.

**Statut** : tranches 2 et 3 livrees, instrument tranche 1 valide, mesure de temps livree (Pli 3), these pivot **NON VERIFIEE** par les mesures disponibles.

**References tranche 1** : issue #18446 (instrument livre par la PR #18639) ; carnet connexe `GOL-08-Loi-II-Translateur-Life.ipynb`.


## Genealogie

**Predecesseur** : [GOL-08](GOL-08-Loi-II-Translateur-Life.ipynb) (Loi II, seconde jambe : synthetiser un translateur Life). Le present carnet prend la releve sur le meme substrat Conway mais transpose le probleme verify-once (verificateur de validite d'un translateur) vers le probleme measure-once (instrument K_trajectory qui mesure l'entropie LZ d'une trajectoire fenetree).

**Tranche 1** : [PR #18639](https://github.com/jsboige/CoursIA/pull/18639) -- instrument `scripts/hashlife/k_trajectory.py` + resultats bruts `scripts/hashlife/k_trajectory_results.json` (7 trajectoires, 5 fenetres).

**Voir #18446** pour le contexte T12 (Lean, #13483).

## Exercices

**Exercice 1 -- Sensibilite au seuil.**

Le discriminant tranche 1 utilise le seuil `0.7` pour separer `PROGRAM-PERIODIC-COLLAPSED` de `SOUP-FRAGILE-WEAK`. Reproduisez le calcul pour le seuil `0.5` et `0.9`. Combien de trajectoires changent de classe ? Le seuil `0.7` est-il robuste ?

**Exercice 2 -- Normalisation par la dimension.**

Chaque trajectoire est codee en flux d'octets de dimensions differentes (glider 16x16, OTCA 7578x7578). K_trajectory depend-il de la dimension ? Proposez une normalisation (par exemple `K_trajectory(t, W) / (rows * W)`) et observez comment le discriminant evolue.

**Exercice 3 -- Compresseur alternatif.**

L'instrument utilise zlib (LZ77). Si on dispose d'un compresseur PPM (par exemple `ppmd-py` ou `zppy`), la borne superieure de Kolmogorov est-elle plus serree ? Tracez le quotient `K_PPM(t, W) / K_LZ(t, W)` sur les 7 trajectoires. La discrimination periodique-court vs entropie tient-elle ?

**Exercice 4 -- Un `AMBIGUOUS` est-il stable ?**

Les deux soupes tombent entre les seuils (`alpha = 0.571` et `0.675`, pour `REDUCIBLE < 0.5` et `IRREDUCIBLE >= 0.8`). Rejouez la mesure avec davantage de repetitions :

```
python scripts/hashlife/hashlife_speedup.py --motif soup-64-d30 --motif soup-128-d35 --repeats 10 --json-out soupes.json
```

Comparez les exposants obtenus a ceux de `scripts/hashlife/hashlife_speedup_results.json`. Le verdict change-t-il ? A quelle distance d'un seuil un `AMBIGUOUS` cesse-t-il d'etre une mesure pour devenir une indifference ?


## References

Carnet 20e (Perplexité Structurelle Hashlife) s'inscrit dans le palier 20 du R2 carnet References (#19461) : 3 acquis → 7 SOTA. **Mapping mesuré** (acquis présents dans le carnet → SOTA qui les couvre ou les prolonge) :

| Acquis (carnet) | SOTA couvrante | Couverture |
|---|---|---|
| Substrat Conway `Game of Life` 2D, règles B3/S23, transition par quartet | **Gardner 1970** -- *Mathematical Games: The fantastic combinations of John Conway's new solitaire game "Life"*, Scientific American 223(4):120–123 (chronique de Gardner, qui presente le jeu de Conway au public) | Le substrat défini par sa table de transition ; c'est le plan de référence du carnet |
| Universaux Life (glider, Gosper gun) et preuves de Turing-completesse | **Berlekamp-Conway-Guy 1982** -- *Winning Ways for your Mathematical Plays*, vol. 2 (chap. 25 "What Is Life?"), Academic Press | Cadre combinatoire des universaux, démonstration que Life est Turing-complet ; le carnet 20e hérite de ce résultat via OTCA metapixel |
| Classes de comportements (périodiques / spaceship / still life / methuselah) et leur catégorisation par complexité | **Wolfram 2002** -- *A New Kind of Science*, Wolfram Media (chap. 5 "Two Dimensions and Beyond", règles 9 étudiées) | Taxonomie empirique des classes de comportement CA 2D ; source canonique pour classifier la sortie `K_trajectory` du carnet |
| Construction du métapixel OTCA (Brice Due, 2005–2006), automate 2048×2048 simulant n'importe quel CA par inclusion | **Brice Due 2006** -- *OTCA metapixel*, page LifeWiki « OTCA metapixel » (construction attestée automne 2005–printemps 2006 par Brice Due ; cellule du carnet §5 cite « OTCA metapixel (Brice Due, 2006) » comme source canonique) | La brique de la Turing-completesse pratique : c'est le composant que Rendell 2010/2011 assemble en UTM, et que le test Turing-complet du carnet vise à mesurer |
| Construction d'un ordinateur de Turing (UTM) en Life par assemblage d'éléments de bas niveau (slugs, gliders, OTCA) | **Rendell 2010/2011** -- *A Turing Machine in Conway's Game of Life*, rendell-attic.org/gol/tm.htm (machine 2000, UTM 2010) + chap. 13 « Turing Universality of the Game of Life » in A. Adamatzky (ed.), *Collision-Based Computing*, Springer | Construction documentée d'un UTM en Life, brique par brique ; le test Turing-complet du carnet s'inspire de la méthode d'assemblage élémentaire |
| Hasard algorithmique (perplexité LZ, Kolmogorov approché) appliqué aux trajectoires CA | **Li-Vitányi 2008** -- *An Introduction to Kolmogorov Complexity and Its Applications* (3e éd.), Springer | Cadre théorique de `K(x)`, méthodes d'approximation LZ ; c'est le fondement de l'instrument `K_trajectory.py` tranche 3 du carnet |
| Algorithme Hashlife (hash-consing + quadtree récursif, simulation exponentiellement rapide) | **Gosper 1984** -- « Exploiting Regularities in Large Cellular Spaces », *Physica* 10D:75-80 (publication primaire, citée par cellule 10 §5 du carnet) + manuscrit de suivi « Hashing Life » (Gosper, ~1994, Stanford) | Source primaire de l'algorithme publié ; le manuscrit 1994 est un suivi technique post-publication. Sans Gosper 1984, la simulation `t=2^k` du carnet serait infaisable |

**Acquis couverts hors-mapping** : `B3/S23` règles (Gardner 1970), taxonomie Wolfram 2002 chap. 5, métapixel OTCA (Brice Due 2006), `K_trajectory` (Li-Vitányi 2008 sec. 2-4), `K(W=1)/K(W=16)` discriminant (Gardner 1970 + Wolfram 2002 chap. 5), borne LZ (Li-Vitányi 2008 sec. 4.4).

**7 SOTA = 7 angles distincts** : substrat définition (Gardner 1970), universaux et Turing-completesse (Berlekamp-Conway-Guy 1982), taxonomie empirique (Wolfram 2002), brique OTCA (Brice Due 2006), construction d'UTM (Rendell 2010/2011), fondement théorique complexité (Li-Vitányi 2008), algorithme d'implémentation (Gosper 1984 + Hashlife 1994). **Couverture = 7/7 = complète** pour la thématique du carnet.
